# Robustez del esquema de partición
### AVM · Distrito Centro (Rosario)

Reproduce el **Cuadro 3.1 / §4.7.2**: sensibilidad del desempeño al esquema de validación, con el **mismo
modelo y variables** (*histogram-based gradient boosting*), bajo tres particiones de severidad creciente:
aleatoria por aviso (cota optimista), agrupada por cuadra/edificio (`building_cluster_id`, protocolo
principal) y agrupada por celda espacial ~500 m (cota exigente).


In [1]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
ROOT = Path.cwd()
while not (ROOT/"data"/"processed").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sns.set_theme(style="whitegrid"); plt.rcParams.update({"figure.dpi":120})
TAB=ROOT/"reports"/"tables"/"modelado"; FIG=ROOT/"reports"/"figures"/"resultados"


## Desempeño por esquema de partición


In [2]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
df=pd.read_csv(ROOT/"data"/"processed"/"zonaprop_v2_matriz_modelado_centro.csv.gz")
yv=pd.to_numeric(df["eda_precio_usd"],errors="coerce").values.astype(float); ytil=np.log1p(yv)
groups=df["building_cluster_id"].values
lat=pd.to_numeric(df["eda_latitud"],errors="coerce").values; lon=pd.to_numeric(df["eda_longitud"],errors="coerce").values
cell=(np.round(lat/0.0045)*1000+np.round(lon/0.0045)).astype(str)   # celda espacial ~500 m
CAT=[c for c in ["eda_antiguedad_estado","eda_disposicion","eda_orientacion","eda_tipo_operacion","barrio_oficial_geo"] if c in df.columns]
drop=["eda_precio_usd","eda_precio_m2","property_id","building_cluster_id","fe_version"]; num=[c for c in df.columns if c not in drop+CAT]
X=df[num+CAT].copy()
for c in num: X[c]=pd.to_numeric(X[c],errors="coerce")
def mk(): return Pipeline([("prep",ColumnTransformer([("num",SimpleImputer(strategy="median"),num),("cat",Pipeline([("i",SimpleImputer(strategy="most_frequent")),("o",OneHotEncoder(handle_unknown="ignore",min_frequency=20))]),CAT)])),("m",HistGradientBoostingRegressor(random_state=42))])
def evalcv(splits):
    r2s=[];rm=[]
    for tr,va in splits:
        p=mk().fit(X.iloc[tr],ytil[tr]); pr=p.predict(X.iloc[va])
        r2s.append(r2_score(ytil[va],pr)); rm.append(np.sqrt(mean_squared_error(yv[va],np.expm1(pr))))
    return np.mean(r2s),np.mean(rm)
res={"Aleatoria, por aviso (referencia)":evalcv(list(KFold(5,shuffle=True,random_state=42).split(X))),
     "Agrupada por cuadra/edificio (principal)":evalcv(list(GroupKFold(5).split(X,ytil,groups))),
     "Agrupada por celda espacial (~500 m)":evalcv(list(GroupKFold(5).split(X,ytil,cell)))}
tab=pd.DataFrame([{"Esquema de partición":k,"R2_log":round(r2,3),"RMSE_USD":round(rm)} for k,(r2,rm) in res.items()])
tab.to_csv(TAB/"robustez_particion.csv",index=False); display(tab)
d=res["Aleatoria, por aviso (referencia)"][0]-res["Agrupada por cuadra/edificio (principal)"][0]
print(f"Brecha aleatoria -> cuadra: {d:.3f} de R² (optimismo acotado de la partición aleatoria).")


,Esquema de partición,R2_log,RMSE_USD
0,"Aleatoria, por aviso (referencia)",0.909,36723
1,Agrupada por cuadra/edificio (principal),0.896,41037
2,Agrupada por celda espacial (~500 m),0.891,42004


Brecha aleatoria -> cuadra: 0.013 de R² (optimismo acotado de la partición aleatoria).


**Lectura.** El mayor salto ocurre al pasar de la partición aleatoria a la agrupada por cuadra (optimismo
acotado, ~0,01–0,02 de R²); el paso a celdas ~500 m agrega una diferencia marginal. La partición agrupada
se adopta como **protocolo principal** por ser la evaluación más conservadora, y la aleatoria se reporta
como **referencia**.
